# 03b - RM-b: frozen encoder + classification head

Encoder IndoBERT dibekukan dan hanya classification head yang dilatih. Embedding
mean-pool diekstrak sekali lalu dipakai ulang oleh seluruh konfigurasi head,
sehingga satu epoch hanya berupa perkalian matriks kecil.

Notebook ini menjalankan KAMPANYE PENUH RM-b: seluruh grid di
`tuning_grids/RMB_TUNING_GRID*.csv`. Setiap konfigurasi head disimpan di
`checkpoints/rmb_heads/` (bukan hanya juara), karena `03c_rmc_rac.ipynb`
menguji RAC di atas SETIAP head, tidak hanya head terbaik.

Prasyarat: `03a_rma_finetune.ipynb` sudah dijalankan (folder keluaran yang sama
dipakai bersama, `outputs/tuning/`).


In [ ]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])


## 1. Ekstraksi fitur beku

In [ ]:
features = runner.features

print(f"dari cache      : {features.from_cache}")
print(f"dimensi         : {features.hidden_dim}")
print(f"waktu ekstraksi : {features.extract_time_s:.2f} s")
print(f"peak GPU memory : {features.extract_peak_mem_mb:.0f} MB")
for split in ("train", "val", "test"):
    embeddings, labels = features[split]
    print(f"  {split:5s}: {embeddings.shape} | label {labels.shape}")


Ekstraksi berjalan sekali per encoder lalu di-cache ke
`outputs/tuning/features/<nama_encoder>/`. Biaya ini tetap dihitung sebagai
bagian waktu latih RM-b agar perbandingannya dengan RM-a jujur, walau
diamortisasi ke seluruh konfigurasi head yang dicoba di bawah.


## 2. Konfigurasi default

Default mengikuti Peters dkk. (2019) untuk transfer berbasis fitur: head
diinisialisasi acak sehingga butuh learning rate lebih tinggi daripada RM-a.
Grid di bawah menyapu kapasitas head, `lr`, `epochs`, dan `dropout` di sekitar
default ini.


In [ ]:
from src.models.schemas import RMBConfig

print(RMBConfig().model_dump())


## 3. Muat rancangan grid

In [ ]:
import pandas as pd

GRID_DIR = settings.data_dir.parent / "tuning_grids"
RMB_GRIDS = ("RMB_TUNING_GRID.csv", "RMB_TUNING_GRID_STAGE1B.csv",
             "RMB_TUNING_GRID_STAGE2.csv", "RMB_TUNING_GRID_STAGE3.csv")


def muat_grid(nama: str) -> list[dict]:
    frame = pd.read_csv(GRID_DIR / nama)
    catatan = frame.pop("catatan") if "catatan" in frame.columns else ""
    return [
        {"config": {k: v for k, v in baris.items() if pd.notna(v)},
         "note": catatan.iloc[i] if hasattr(catatan, "iloc") else ""}
        for i, baris in enumerate(frame.to_dict("records"))
    ]


for berkas in RMB_GRIDS:
    print(f"  {berkas}: {len(pd.read_csv(GRID_DIR / berkas))} konfigurasi")


## Melanjutkan kampanye yang terputus

Sama seperti 03a: `run_batch` melewati konfigurasi yang sudah tercatat
(`resume=True`), sehingga sel di bawah aman dijalankan ulang apa adanya
setelah kernel mati atau proses dihentikan.


In [ ]:
permintaan_rmb = [item for berkas in RMB_GRIDS for item in muat_grid(berkas)]
tersisa = runner.pending_requests("rmb", permintaan_rmb)
print(f"{len(permintaan_rmb) - len(tersisa)}/{len(permintaan_rmb)} selesai, {len(tersisa)} tersisa")


## 4. Jalankan seluruh grid

In [ ]:
batch_rmb = []
for berkas in RMB_GRIDS:
    batch_id = berkas.replace(".csv", "").lower()
    runner.run_batch("rmb", muat_grid(berkas), batch_id=batch_id)
    batch_rmb.append(batch_id)

riwayat_rmb = runner.reporter.runs_frame("rmb")
riwayat_rmb[riwayat_rmb["batch_id"].isin(batch_rmb)].nlargest(10, "val_f1_macro")[
    ["run_id", "head_arch", "hidden_dim", "lr", "epochs", "dropout",
     "val_f1_macro", "train_time_s", "trainable_params"]
]


Setiap konfigurasi RM-b menyimpan state head-nya di
`outputs/tuning/checkpoints/rmb_heads/run_{id}.pt`. Head itulah yang diuji di
eksplorasi RM-c pada `03c_rmc_rac.ipynb`, jadi head yang dieksplorasi sama
persis dengan yang dilatih di sini.


## 5. Kurva epoch juara

In [ ]:
import json

best_rmb = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))["rmb"]
print(f"juara RM-b: run #{best_rmb['run_id']} | val F1-macro {best_rmb['val_f1_macro']:.4f}")
print(f"trainable params: {best_rmb['trainable_params']:,}")

history = pd.read_csv(OUT_DIR / "history" / "rmb_history.csv")
history[history["run_id"] == best_rmb["run_id"]]


## Ringkasan

Juara RM-b tercatat di `best.json`; checkpoint-nya di
`checkpoints/rmb_best.pt` dan seluruh head individual di
`checkpoints/rmb_heads/`. Perbandingan trainable parameter terhadap RM-a
adalah inti klaim efisiensi skenario ini.

Lanjut ke `03c_rmc_rac.ipynb`.
